
# FusionConnect AI: Physics & Nuclear Fusion Streamlit Platform

This notebook builds a **single-file Streamlit prototype** for a public physics and nuclear-fusion platform founded and authored by **Ethan Meline**, with **Dr. Qingyang Xiao** serving as advisor. It includes:

- Public education modules for physics and nuclear fusion basics.
- Anonymous onboarding and consent-aware profile collection.
- Reddit/X-style community posts, comments, and upvotes.
- Research/classroom collaboration proposals.
- AI mentor recommendations using supervised machine learning, a lightweight neural network, and feedback-based reinforcement-learning logic.
- Founder dashboard with anonymized headcount, engagement analytics, and QR-code generation.

The notebook writes the deployable files `fusionconnect_ai_streamlit_app.py`, `requirements.txt`, and `.streamlit/config.toml`. You can run the app from Colab, locally, or deploy it on Streamlit Community Cloud.


**Founder & Author:** Ethan Meline  
**Advisor:** Dr. Qingyang Xiao



## Important production notes

This is a working prototype, not a finished production social network. Before launching publicly, add:

1. A real privacy policy, terms of use, moderation policy, and consent process, especially if minors may use it.
2. Real authentication, admin roles, and a persistent managed database such as PostgreSQL, Supabase, or Firebase.
3. Expert review of the educational content and safety review of community features.
4. A deployment workflow using GitHub + Streamlit Community Cloud.

The included SQLite database is fine for a demo, but public deployments need persistent external storage because free app runtimes can restart.


In [ ]:

# 1) Check Python version and install dependencies.
# In Google Colab, run this cell first. It is designed for Python 3.12.x.
import sys
print('Python version:', sys.version)

# Install packages. Restart runtime only if Colab asks you to.
%pip install -q "streamlit>=1.37" "pandas>=2.2" "numpy>=1.26" "scikit-learn>=1.5" "plotly>=5.22" "qrcode[pil]>=7.4" "python-dateutil>=2.9"


In [ ]:

# 2) Write dependency and Streamlit configuration files.
from pathlib import Path

Path('.streamlit').mkdir(exist_ok=True)
Path('requirements.txt').write_text('streamlit>=1.37\npandas>=2.2\nnumpy>=1.26\nscikit-learn>=1.5\nplotly>=5.22\nqrcode[pil]>=7.4\npython-dateutil>=2.9\n', encoding='utf-8')
Path('.streamlit/config.toml').write_text('[theme]\nbase = "light"\nprimaryColor = "#3867D6"\nbackgroundColor = "#FFFFFF"\nsecondaryBackgroundColor = "#F5F7FB"\ntextColor = "#111827"\n\n[server]\nheadless = true\nenableCORS = false\n', encoding='utf-8')

print('Created requirements.txt and .streamlit/config.toml')
print(Path('requirements.txt').read_text())


In [ ]:
# 3) Write the complete Streamlit app to a Python file.
from pathlib import Path

app_code = '# fusionconnect_ai_streamlit_app.py\n# A Streamlit prototype for an AI-assisted physics and nuclear-fusion learning/community platform.\n# Built for Python 3.12+ and designed to run from Google Colab, locally, or Streamlit Community Cloud.\n\nfrom __future__ import annotations\n\nimport base64\nimport io\nimport json\nimport os\nimport random\nimport sqlite3\nimport textwrap\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport qrcode\nimport streamlit as st\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\nAPP_NAME = \'FusionConnect AI\'\nFOUNDER_NAME = \'Ethan Meline\'\nADVISOR_NAME = \'Dr. Qingyang Xiao\'\nAPP_DIR = Path(__file__).resolve().parent\nDATA_DIR = APP_DIR / \'data\'\nDATA_DIR.mkdir(exist_ok=True)\nDB_PATH = Path(os.getenv(\'FUSIONCONNECT_DB_PATH\', str(DATA_DIR / \'fusionconnect_ai.sqlite3\')))\n\nTOPICS = [\n    \'fusion basics\', \'plasma physics\', \'tokamak\', \'stellarator\', \'inertial fusion\',\n    \'magnetic confinement\', \'materials\', \'tritium breeding\', \'diagnostics\',\n    \'AI for fusion\', \'nuclear safety\', \'energy policy\', \'education\', \'research collaboration\'\n]\n\nROLES = [\'Student\', \'Teacher\', \'Researcher\', \'Engineer\', \'Industry\', \'Policy / Public\', \'Curious Public\']\nKNOWLEDGE_LEVELS = [\'Beginner\', \'Intermediate\', \'Advanced\']\nAGE_BANDS = [\'Prefer not to say\', \'Under 13\', \'13-17\', \'18-24\', \'25-34\', \'35-44\', \'45-54\', \'55+\']\nEDUCATION_LEVELS = [\'Prefer not to say\', \'Middle school\', \'High school\', \'Undergraduate\', \'Graduate\', \'Professional\', \'Self learner\']\nGOALS = [\n    \'Learn fundamentals\', \'Prepare a class\', \'Find research collaborators\', \'Explore project ideas\',\n    \'Track fusion progress\', \'Share outreach content\', \'Build AI/ML skills\', \'Ask questions\'\n]\n\nLEARNING_MODULES = [\n    {\n        \'id\': \'fusion_intro\',\n        \'title\': \'Fusion in One Page\',\n        \'level\': \'Beginner\',\n        \'tags\': [\'fusion basics\', \'energy policy\', \'education\'],\n        \'summary\': \'What fusion is, why it releases energy, and why it is hard to build on Earth.\',\n        \'content\': """\nFusion joins light atomic nuclei into heavier nuclei. The classic fuel pair is deuterium and tritium, two hydrogen isotopes. When they fuse, they produce helium, a neutron, and energy. The energy comes from the difference between the initial and final nuclear binding energy.\n\nThe core challenge is not the reaction itself. The challenge is creating and controlling a plasma hot and dense enough, for long enough, that useful fusion reactions happen faster than the plasma loses energy.\n\nThree big engineering questions guide most fusion projects:\n\n1. Can the plasma reach sufficient temperature, density, and confinement time?\n2. Can the reactor materials survive heat, neutron damage, and repeated operation?\n3. Can the system produce fuel, extract heat, and run economically?\n""",\n        \'activity\': \'Draw a three-circle diagram: temperature, density, and confinement time. Put “net useful energy” where they overlap.\',\n        \'quiz_question\': \'Which three plasma conditions are commonly combined in fusion performance discussions?\',\n        \'quiz_answer\': \'temperature, density, and confinement time\'\n    },\n    {\n        \'id\': \'plasma_basics\',\n        \'title\': \'Plasma Physics Basics\',\n        \'level\': \'Beginner\',\n        \'tags\': [\'plasma physics\', \'magnetic confinement\', \'education\'],\n        \'summary\': \'Ionized gas, charged particles, magnetic fields, and collective behavior.\',\n        \'content\': """\nA plasma is an ionized gas containing free electrons and ions. Because charged particles respond to electric and magnetic fields, plasma can behave collectively rather than like ordinary neutral gas.\n\nA strong magnetic field can make charged particles spiral around field lines. In magnetic confinement devices, this helps keep the hot plasma away from solid walls.\n\nImportant vocabulary:\n\n- Ion: an atom or molecule with missing or extra electrons.\n- Electron temperature: a measure of average electron kinetic energy.\n- Debye shielding: the tendency of plasma to screen electric fields over short distances.\n- Instability: a growing disturbance that can degrade confinement.\n""",\n        \'activity\': \'Use a pencil to trace a spiral around a line. That line represents a magnetic field line.\',\n        \'quiz_question\': \'Why do magnetic fields help in a fusion plasma?\',\n        \'quiz_answer\': \'charged particles spiral around magnetic field lines, helping confinement\'\n    },\n    {\n        \'id\': \'tokamak_overview\',\n        \'title\': \'Tokamak Overview\',\n        \'level\': \'Intermediate\',\n        \'tags\': [\'tokamak\', \'magnetic confinement\', \'diagnostics\'],\n        \'summary\': \'A doughnut-shaped magnetic bottle for high-temperature plasma.\',\n        \'content\': """\nA tokamak confines plasma in a torus, similar to a doughnut shape. Magnetic coils produce a strong toroidal field, and a plasma current helps create a poloidal field. Together they form helical magnetic field lines that improve confinement.\n\nTokamak design topics include:\n\n- Plasma current drive and transformer action.\n- Divertors that exhaust heat and particles.\n- Edge-localized modes and disruptions.\n- Diagnostics for temperature, density, impurities, and magnetic behavior.\n""",\n        \'activity\': \'Sketch a doughnut and draw arrows around the long way and short way to represent toroidal and poloidal directions.\',\n        \'quiz_question\': \'What is the approximate shape of a tokamak plasma chamber?\',\n        \'quiz_answer\': \'a torus or doughnut shape\'\n    },\n    {\n        \'id\': \'stellarator_overview\',\n        \'title\': \'Stellarator Overview\',\n        \'level\': \'Intermediate\',\n        \'tags\': [\'stellarator\', \'magnetic confinement\', \'research collaboration\'],\n        \'summary\': \'A twisted magnetic confinement device that can operate without large plasma current.\',\n        \'content\': """\nA stellarator uses carefully shaped external coils to create twisted magnetic fields. This can reduce reliance on large plasma current, which may support steady-state operation.\n\nStellarators are mathematically and mechanically complex. Modern optimization and high-performance computing have made better stellarator designs possible.\n\nKey ideas:\n\n- Magnetic surface optimization.\n- Complex coil geometry.\n- Reduced disruption risk compared with current-driven devices.\n- Engineering challenges in coil manufacturing and maintenance.\n""",\n        \'activity\': \'Compare a simple circular coil with a twisted 3D coil. Ask: which is easier to build, and which gives more magnetic control?\',\n        \'quiz_question\': \'What is one potential advantage of stellarators?\',\n        \'quiz_answer\': \'steady-state operation with less need for large plasma current\'\n    },\n    {\n        \'id\': \'inertial_fusion\',\n        \'title\': \'Inertial Fusion Basics\',\n        \'level\': \'Intermediate\',\n        \'tags\': [\'inertial fusion\', \'fusion basics\', \'diagnostics\'],\n        \'summary\': \'Compressing tiny fuel targets with lasers or particle beams.\',\n        \'content\': """\nIn inertial confinement fusion, a small fuel target is rapidly compressed and heated. The fuel inertia holds the material together briefly while fusion reactions occur.\n\nThe target must be extremely symmetric. Small imperfections can grow into instabilities and reduce performance.\n\nImportant ideas:\n\n- Implosion symmetry.\n- Target capsule design.\n- Laser or beam energy coupling.\n- Diagnostics for neutron yield, temperature, and compression.\n""",\n        \'activity\': \'Imagine squeezing a balloon evenly from all sides. What happens if one side is pushed harder than another?\',\n        \'quiz_question\': \'Why is symmetry important in inertial fusion?\',\n        \'quiz_answer\': \'asymmetry can seed instabilities and reduce compression\'\n    },\n    {\n        \'id\': \'lawson_q\',\n        \'title\': \'Lawson Criterion and Fusion Gain\',\n        \'level\': \'Advanced\',\n        \'tags\': [\'fusion basics\', \'plasma physics\', \'AI for fusion\'],\n        \'summary\': \'How density, temperature, and confinement time connect to reactor performance.\',\n        \'content\': """\nThe Lawson criterion expresses a condition for fusion power to exceed losses. In practice, researchers often discuss the triple product: density × temperature × confinement time.\n\nFusion gain Q is the ratio of fusion power produced to external heating power delivered to the plasma. Q is useful, but it is not the same as net electricity from a complete power plant. A power plant must also handle energy conversion, recirculating power, tritium breeding, maintenance, and economics.\n\nAI can help by predicting confinement behavior, detecting instability precursors, optimizing control, and discovering patterns in high-dimensional diagnostic data.\n""",\n        \'activity\': \'Make a table with columns for density, temperature, confinement time, and Q. Fill in hypothetical devices and discuss tradeoffs.\',\n        \'quiz_question\': \'What is the fusion triple product?\',\n        \'quiz_answer\': \'density times temperature times confinement time\'\n    },\n    {\n        \'id\': \'materials_tritium\',\n        \'title\': \'Materials and Tritium Breeding\',\n        \'level\': \'Advanced\',\n        \'tags\': [\'materials\', \'tritium breeding\', \'nuclear safety\'],\n        \'summary\': \'Why a fusion power plant is also a materials and fuel-cycle challenge.\',\n        \'content\': """\nA deuterium-tritium fusion reactor produces energetic neutrons. These neutrons carry energy to the blanket, where heat can be extracted. They also damage materials and can help breed tritium from lithium.\n\nMaterials must handle neutron damage, high heat flux, thermal cycling, corrosion, and maintainability. Tritium handling requires careful accounting, containment, and safety practices.\n\nEngineering success depends on the full system, not only the plasma.\n""",\n        \'activity\': \'List three reactor subsystems outside the plasma that must work for a commercial plant.\',\n        \'quiz_question\': \'Why is lithium important in many fusion blanket concepts?\',\n        \'quiz_answer\': \'lithium can breed tritium when interacting with neutrons\'\n    },\n    {\n        \'id\': \'ai_fusion\',\n        \'title\': \'AI for Fusion Research\',\n        \'level\': \'Intermediate\',\n        \'tags\': [\'AI for fusion\', \'diagnostics\', \'research collaboration\'],\n        \'summary\': \'How ML, neural networks, and reinforcement learning can support fusion research.\',\n        \'content\': """\nAI can support fusion in several ways:\n\n- Supervised learning: predict plasma state, classify events, or recommend content based on labeled examples.\n- Deep neural networks: learn nonlinear patterns in diagnostic data, images, and time series.\n- Reinforcement learning: learn control or recommendation policies from feedback and rewards.\n\nFor this app, the AI recommender is intentionally transparent. It combines user profile interests, app behavior, supervised learning, a small neural network, and feedback-based bandit rewards.\n""",\n        \'activity\': \'Choose a fusion problem and label it as classification, regression, clustering, or reinforcement learning.\',\n        \'quiz_question\': \'What kind of learning uses feedback rewards to improve future actions?\',\n        \'quiz_answer\': \'reinforcement learning\'\n    }\n]\n\nACTIONS = [\n    {\'action_id\': \'learn_fusion_intro\', \'label\': \'Start with Fusion in One Page\', \'type\': \'learn\', \'item_id\': \'fusion_intro\', \'tags\': [\'fusion basics\', \'education\'], \'why\': \'Build a clear foundation before advanced topics.\'},\n    {\'action_id\': \'learn_plasma_basics\', \'label\': \'Study Plasma Physics Basics\', \'type\': \'learn\', \'item_id\': \'plasma_basics\', \'tags\': [\'plasma physics\', \'magnetic confinement\'], \'why\': \'Plasma behavior is central to most fusion concepts.\'},\n    {\'action_id\': \'learn_tokamak\', \'label\': \'Explore Tokamak Overview\', \'type\': \'learn\', \'item_id\': \'tokamak_overview\', \'tags\': [\'tokamak\', \'diagnostics\'], \'why\': \'Tokamaks are a major path in magnetic confinement fusion.\'},\n    {\'action_id\': \'learn_stellarator\', \'label\': \'Explore Stellarator Overview\', \'type\': \'learn\', \'item_id\': \'stellarator_overview\', \'tags\': [\'stellarator\', \'magnetic confinement\'], \'why\': \'Stellarators show how optimized 3D fields can support confinement.\'},\n    {\'action_id\': \'learn_inertial\', \'label\': \'Learn Inertial Fusion Basics\', \'type\': \'learn\', \'item_id\': \'inertial_fusion\', \'tags\': [\'inertial fusion\', \'diagnostics\'], \'why\': \'Inertial fusion is a distinct path with different physics and engineering constraints.\'},\n    {\'action_id\': \'learn_lawson\', \'label\': \'Study Lawson Criterion and Fusion Gain\', \'type\': \'learn\', \'item_id\': \'lawson_q\', \'tags\': [\'fusion basics\', \'plasma physics\', \'AI for fusion\'], \'why\': \'This topic connects plasma performance to reactor goals.\'},\n    {\'action_id\': \'learn_materials\', \'label\': \'Read Materials and Tritium Breeding\', \'type\': \'learn\', \'item_id\': \'materials_tritium\', \'tags\': [\'materials\', \'tritium breeding\', \'nuclear safety\'], \'why\': \'Commercial fusion depends on robust materials and fuel-cycle design.\'},\n    {\'action_id\': \'learn_ai_fusion\', \'label\': \'Learn AI for Fusion Research\', \'type\': \'learn\', \'item_id\': \'ai_fusion\', \'tags\': [\'AI for fusion\', \'diagnostics\'], \'why\': \'AI can help connect education, diagnostics, prediction, control, and collaboration.\'},\n    {\'action_id\': \'write_intro_post\', \'label\': \'Share your first idea or question\', \'type\': \'community\', \'item_id\': \'new_post\', \'tags\': [\'education\', \'research collaboration\'], \'why\': \'Posting helps other users find your interests and start discussion.\'},\n    {\'action_id\': \'join_collab\', \'label\': \'Browse collaboration proposals\', \'type\': \'collaboration\', \'item_id\': \'collab_board\', \'tags\': [\'research collaboration\', \'AI for fusion\'], \'why\': \'Collaboration is often the fastest way to turn curiosity into a project.\'},\n    {\'action_id\': \'create_collab\', \'label\': \'Create a mini research collaboration proposal\', \'type\': \'collaboration\', \'item_id\': \'new_collab\', \'tags\': [\'research collaboration\', \'AI for fusion\'], \'why\': \'A clear project idea can attract students, teachers, or researchers.\'},\n    {\'action_id\': \'take_quiz\', \'label\': \'Take a quick learning quiz\', \'type\': \'learn\', \'item_id\': \'quiz\', \'tags\': [\'education\', \'fusion basics\'], \'why\': \'Quizzes convert passive reading into active learning.\'}\n]\n\nBAD_WORDS = {\'spam\', \'scam\', \'hate\', \'violent threat\'}\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat(timespec=\'seconds\')\n\n\ndef safe_json_loads(value: Any, default: Any) -> Any:\n    if value is None:\n        return default\n    if isinstance(value, (list, dict)):\n        return value\n    try:\n        return json.loads(value)\n    except Exception:\n        return default\n\n\ndef json_dumps(value: Any) -> str:\n    return json.dumps(value, ensure_ascii=False, sort_keys=True)\n\n\ndef get_conn() -> sqlite3.Connection:\n    conn = sqlite3.connect(DB_PATH, check_same_thread=False, timeout=30)\n    conn.row_factory = sqlite3.Row\n    return conn\n\n\ndef run_sql(query: str, params: Tuple[Any, ...] = ()) -> None:\n    with get_conn() as conn:\n        conn.execute(query, params)\n        conn.commit()\n\n\ndef query_df(query: str, params: Tuple[Any, ...] = ()) -> pd.DataFrame:\n    with get_conn() as conn:\n        return pd.read_sql_query(query, conn, params=params)\n\n\ndef query_one(query: str, params: Tuple[Any, ...] = ()) -> Optional[sqlite3.Row]:\n    with get_conn() as conn:\n        cur = conn.execute(query, params)\n        return cur.fetchone()\n\n\ndef init_db() -> None:\n    schema = [\n        \'\'\'CREATE TABLE IF NOT EXISTS users (\n            user_id TEXT PRIMARY KEY,\n            created_at TEXT NOT NULL,\n            last_seen TEXT,\n            display_name TEXT,\n            role TEXT,\n            age_band TEXT,\n            education_level TEXT,\n            region TEXT,\n            knowledge_level TEXT,\n            interests_json TEXT,\n            goals_json TEXT,\n            consent_ai INTEGER DEFAULT 0,\n            consent_public INTEGER DEFAULT 0,\n            consent_research INTEGER DEFAULT 0,\n            referral_source TEXT\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS events (\n            event_id TEXT PRIMARY KEY,\n            user_id TEXT,\n            event_type TEXT,\n            item_id TEXT,\n            topic TEXT,\n            metadata_json TEXT,\n            created_at TEXT NOT NULL\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS posts (\n            post_id TEXT PRIMARY KEY,\n            user_id TEXT,\n            display_name TEXT,\n            category TEXT,\n            title TEXT,\n            body TEXT,\n            tags_json TEXT,\n            upvotes INTEGER DEFAULT 0,\n            created_at TEXT NOT NULL\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS comments (\n            comment_id TEXT PRIMARY KEY,\n            post_id TEXT,\n            user_id TEXT,\n            display_name TEXT,\n            body TEXT,\n            created_at TEXT NOT NULL\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS collaborations (\n            collab_id TEXT PRIMARY KEY,\n            user_id TEXT,\n            display_name TEXT,\n            title TEXT,\n            summary TEXT,\n            topics_json TEXT,\n            skills_needed TEXT,\n            contact_hint TEXT,\n            status TEXT,\n            created_at TEXT NOT NULL\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS feedback (\n            feedback_id TEXT PRIMARY KEY,\n            user_id TEXT,\n            action_id TEXT,\n            suggested_item TEXT,\n            rating INTEGER,\n            accepted INTEGER,\n            reason TEXT,\n            created_at TEXT NOT NULL\n        )\'\'\',\n        \'\'\'CREATE TABLE IF NOT EXISTS action_rewards (\n            segment TEXT,\n            action_id TEXT,\n            shown_count INTEGER DEFAULT 0,\n            reward_sum REAL DEFAULT 0,\n            last_updated TEXT,\n            PRIMARY KEY (segment, action_id)\n        )\'\'\'\n    ]\n    with get_conn() as conn:\n        for stmt in schema:\n            conn.execute(stmt)\n        conn.commit()\n\n\ndef log_event(user_id: Optional[str], event_type: str, item_id: str = \'\', topic: str = \'\', metadata: Optional[Dict[str, Any]] = None) -> None:\n    run_sql(\n        \'INSERT INTO events VALUES (?, ?, ?, ?, ?, ?, ?)\',\n        (str(uuid.uuid4()), user_id, event_type, item_id, topic, json_dumps(metadata or {}), utc_now())\n    )\n    if user_id:\n        run_sql(\'UPDATE users SET last_seen = ? WHERE user_id = ?\', (utc_now(), user_id))\n\n\ndef content_is_ok(text: str) -> Tuple[bool, str]:\n    lowered = (text or \'\').lower()\n    for term in BAD_WORDS:\n        if term in lowered:\n            return False, f\'Please revise before posting. The term “{term}” triggered the basic moderation filter.\'\n    if len(lowered.strip()) < 3:\n        return False, \'Please add more detail before posting.\'\n    return True, \'\'\n\n\ndef make_demo_name(user_id: str) -> str:\n    return f\'FusionUser-{user_id[-5:]}\'\n\n\ndef current_referral() -> str:\n    try:\n        ref = st.query_params.get(\'ref\', \'organic\')\n        if isinstance(ref, list):\n            ref = ref[0] if ref else \'organic\'\n        return str(ref)[:80]\n    except Exception:\n        return \'organic\'\n\n\ndef create_user_if_needed() -> str:\n    if \'user_id\' not in st.session_state:\n        st.session_state.user_id = str(uuid.uuid4())\n    user_id = st.session_state.user_id\n    existing = query_one(\'SELECT user_id FROM users WHERE user_id = ?\', (user_id,))\n    if not existing:\n        run_sql(\n            \'\'\'INSERT INTO users (\n                user_id, created_at, last_seen, display_name, role, age_band, education_level, region,\n                knowledge_level, interests_json, goals_json, consent_ai, consent_public, consent_research, referral_source\n            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\'\'\',\n            (\n                user_id, utc_now(), utc_now(), make_demo_name(user_id), \'Curious Public\', \'Prefer not to say\',\n                \'Prefer not to say\', \'Prefer not to say\', \'Beginner\', json_dumps([\'fusion basics\']),\n                json_dumps([\'Learn fundamentals\']), 0, 0, 0, current_referral()\n            )\n        )\n        log_event(user_id, \'signup\', topic=\'onboarding\')\n    return user_id\n\n\ndef get_user(user_id: str) -> Dict[str, Any]:\n    row = query_one(\'SELECT * FROM users WHERE user_id = ?\', (user_id,))\n    if not row:\n        raise ValueError(\'User not found\')\n    data = dict(row)\n    data[\'interests\'] = safe_json_loads(data.pop(\'interests_json\', \'[]\'), [])\n    data[\'goals\'] = safe_json_loads(data.pop(\'goals_json\', \'[]\'), [])\n    return data\n\n\ndef update_user_profile(user_id: str, profile: Dict[str, Any]) -> None:\n    run_sql(\n        \'\'\'UPDATE users SET display_name=?, role=?, age_band=?, education_level=?, region=?, knowledge_level=?,\n           interests_json=?, goals_json=?, consent_ai=?, consent_public=?, consent_research=?, last_seen=?\n           WHERE user_id=?\'\'\',\n        (\n            profile[\'display_name\'], profile[\'role\'], profile[\'age_band\'], profile[\'education_level\'],\n            profile[\'region\'], profile[\'knowledge_level\'], json_dumps(profile[\'interests\']), json_dumps(profile[\'goals\']),\n            int(profile[\'consent_ai\']), int(profile[\'consent_public\']), int(profile[\'consent_research\']), utc_now(), user_id\n        )\n    )\n    log_event(user_id, \'profile_saved\', topic=\'onboarding\', metadata={\'interests\': profile[\'interests\'], \'goals\': profile[\'goals\']})\n\n\ndef ensure_seed_data() -> None:\n    marker = query_one("SELECT COUNT(*) AS n FROM events WHERE event_type = \'seed_loaded\'")\n    if marker and marker[\'n\'] > 0:\n        return\n\n    seed_posts = [\n        (\'Teacher prompt: how would you explain plasma to middle-school students?\', \'Education\', [\'education\', \'plasma physics\'], \'Try comparing a plasma to a gas where many particles are electrically charged and can respond together to fields.\'),\n        (\'Question: tokamak vs stellarator tradeoffs\', \'Question\', [\'tokamak\', \'stellarator\'], \'Tokamaks are often simpler geometrically; stellarators move complexity into external coils and optimization.\'),\n        (\'Mini project idea: AI recommender for fusion learning paths\', \'AI / ML\', [\'AI for fusion\', \'education\'], \'A small supervised model can recommend learning modules based on interests and behavior. Feedback can update a bandit policy.\'),\n        (\'Collaboration idea: fusion glossary for beginners\', \'Outreach\', [\'fusion basics\', \'education\'], \'A shared glossary with simple definitions would help new users join discussions faster.\'),\n    ]\n    with get_conn() as conn:\n        for title, category, tags, body in seed_posts:\n            conn.execute(\n                \'INSERT INTO posts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)\',\n                (str(uuid.uuid4()), \'seed\', \'Ethan Meline Demo Team\', category, title, body, json_dumps(tags), random.randint(3, 18), utc_now())\n            )\n        conn.commit()\n\n    # Seed anonymous synthetic examples. These are not real users; they only make the demo AI usable before real traffic arrives.\n    roles = ROLES\n    levels = KNOWLEDGE_LEVELS\n    for i in range(45):\n        uid = f\'seed-user-{i:03d}\'\n        interests = random.sample(TOPICS, k=random.randint(2, 4))\n        goals = random.sample(GOALS, k=random.randint(1, 3))\n        role = random.choice(roles)\n        level = random.choice(levels)\n        run_sql(\n            \'\'\'INSERT OR IGNORE INTO users VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\'\'\',\n            (uid, utc_now(), utc_now(), f\'DemoUser-{i:03d}\', role, random.choice(AGE_BANDS), random.choice(EDUCATION_LEVELS), \'Demo region\', level, json_dumps(interests), json_dumps(goals), 1, 1, 0, \'seed\')\n        )\n        # Choose actions that approximately match the interests.\n        matching = [a for a in ACTIONS if set(a[\'tags\']).intersection(interests)] or ACTIONS\n        for _ in range(random.randint(2, 6)):\n            action = random.choice(matching)\n            rating = random.choice([3, 4, 4, 5, 5])\n            accepted = 1 if rating >= 4 else 0\n            run_sql(\n                \'INSERT INTO feedback VALUES (?, ?, ?, ?, ?, ?, ?, ?)\',\n                (str(uuid.uuid4()), uid, action[\'action_id\'], action[\'item_id\'], rating, accepted, \'synthetic seed\', utc_now())\n            )\n            log_event(uid, random.choice([\'module_view\', \'post_view\', \'collab_view\']), action[\'item_id\'], random.choice(action[\'tags\']))\n\n    run_sql(\'INSERT INTO events VALUES (?, ?, ?, ?, ?, ?, ?)\', (str(uuid.uuid4()), \'system\', \'seed_loaded\', \'\', \'\', \'{}\', utc_now()))\n\n\ndef user_event_counts() -> pd.DataFrame:\n    events = query_df(\'SELECT user_id, event_type, COUNT(*) AS n FROM events GROUP BY user_id, event_type\')\n    if events.empty:\n        return pd.DataFrame(columns=[\'user_id\'])\n    pivot = events.pivot_table(index=\'user_id\', columns=\'event_type\', values=\'n\', aggfunc=\'sum\', fill_value=0).reset_index()\n    pivot.columns = [str(c) for c in pivot.columns]\n    return pivot\n\n\ndef base_feature_frame() -> pd.DataFrame:\n    users = query_df(\'SELECT * FROM users\')\n    if users.empty:\n        return users\n    counts = user_event_counts()\n    df = users.merge(counts, how=\'left\', on=\'user_id\')\n    for col in [\'signup\', \'profile_saved\', \'module_view\', \'post_created\', \'collab_created\', \'recommendation_seen\']:\n        if col not in df.columns:\n            df[col] = 0\n    df[[\'signup\', \'profile_saved\', \'module_view\', \'post_created\', \'collab_created\', \'recommendation_seen\']] = df[[\'signup\', \'profile_saved\', \'module_view\', \'post_created\', \'collab_created\', \'recommendation_seen\']].fillna(0)\n    df[\'interests\'] = df[\'interests_json\'].apply(lambda x: safe_json_loads(x, []))\n    df[\'goals\'] = df[\'goals_json\'].apply(lambda x: safe_json_loads(x, []))\n    for topic in TOPICS:\n        df[f\'interest__{topic}\'] = df[\'interests\'].apply(lambda xs, t=topic: 1 if t in xs else 0)\n    for goal in GOALS:\n        df[f\'goal__{goal}\'] = df[\'goals\'].apply(lambda xs, g=goal: 1 if g in xs else 0)\n    return df\n\n\ndef feature_columns() -> Tuple[List[str], List[str]]:\n    categorical = [\'role\', \'age_band\', \'education_level\', \'region\', \'knowledge_level\']\n    numeric = [\'signup\', \'profile_saved\', \'module_view\', \'post_created\', \'collab_created\', \'recommendation_seen\']\n    numeric += [f\'interest__{topic}\' for topic in TOPICS]\n    numeric += [f\'goal__{goal}\' for goal in GOALS]\n    return categorical, numeric\n\n\ndef train_models() -> Tuple[Optional[Pipeline], Optional[Pipeline], List[str], str]:\n    features = base_feature_frame()\n    feedback = query_df(\'SELECT user_id, action_id, rating, accepted FROM feedback WHERE rating >= 4 OR accepted = 1\')\n    if features.empty or feedback.empty:\n        return None, None, [], \'Not enough data yet. Using transparent rule-based cold-start recommendations.\'\n    df = feedback.merge(features, how=\'inner\', on=\'user_id\')\n    if len(df) < 12 or df[\'action_id\'].nunique() < 2:\n        return None, None, [], \'Not enough labeled feedback yet. Using transparent rule-based cold-start recommendations.\'\n\n    categorical, numeric = feature_columns()\n    for col in categorical:\n        df[col] = df[col].fillna(\'Unknown\').astype(str)\n    for col in numeric:\n        if col not in df.columns:\n            df[col] = 0\n        df[col] = pd.to_numeric(df[col], errors=\'coerce\').fillna(0)\n\n    X = df[categorical + numeric]\n    y = df[\'action_id\'].astype(str)\n    labels = sorted(y.unique().tolist())\n\n    preprocessor = ColumnTransformer(\n        transformers=[\n            (\'cat\', OneHotEncoder(handle_unknown=\'ignore\'), categorical),\n            (\'num\', StandardScaler(with_mean=False), numeric),\n        ]\n    )\n    rf = Pipeline([\n        (\'prep\', preprocessor),\n        (\'model\', RandomForestClassifier(n_estimators=140, random_state=42, class_weight=\'balanced\'))\n    ])\n    rf.fit(X, y)\n\n    mlp = None\n    if len(df) >= 18 and y.nunique() >= 3:\n        mlp = Pipeline([\n            (\'prep\', preprocessor),\n            (\'model\', MLPClassifier(hidden_layer_sizes=(48, 24), activation=\'relu\', random_state=42, max_iter=450, early_stopping=True))\n        ])\n        try:\n            mlp.fit(X, y)\n        except Exception:\n            mlp = None\n\n    return rf, mlp, labels, f\'Trained on {len(df)} positive feedback examples across {len(labels)} actions.\'\n\n\ndef features_for_current_user(user: Dict[str, Any]) -> pd.DataFrame:\n    categorical, numeric = feature_columns()\n    row: Dict[str, Any] = {c: user.get(c, \'Unknown\') for c in categorical}\n    counts = query_df(\'SELECT event_type, COUNT(*) AS n FROM events WHERE user_id = ? GROUP BY event_type\', (user[\'user_id\'],))\n    count_map = dict(zip(counts[\'event_type\'], counts[\'n\'])) if not counts.empty else {}\n    for col in [\'signup\', \'profile_saved\', \'module_view\', \'post_created\', \'collab_created\', \'recommendation_seen\']:\n        row[col] = int(count_map.get(col, 0))\n    interests = set(user.get(\'interests\', []))\n    goals = set(user.get(\'goals\', []))\n    for topic in TOPICS:\n        row[f\'interest__{topic}\'] = 1 if topic in interests else 0\n    for goal in GOALS:\n        row[f\'goal__{goal}\'] = 1 if goal in goals else 0\n    for col in numeric:\n        row.setdefault(col, 0)\n    return pd.DataFrame([row])[categorical + numeric]\n\n\ndef segment_for_user(user: Dict[str, Any]) -> str:\n    return f"{user.get(\'role\',\'Unknown\')}|{user.get(\'knowledge_level\',\'Unknown\')}"\n\n\ndef bandit_score(action_id: str, user: Dict[str, Any]) -> float:\n    segment = segment_for_user(user)\n    row = query_one(\'SELECT shown_count, reward_sum FROM action_rewards WHERE segment = ? AND action_id = ?\', (segment, action_id))\n    global_row = query_one(\'SELECT SUM(shown_count) AS shown_count, SUM(reward_sum) AS reward_sum FROM action_rewards WHERE action_id = ?\', (action_id,))\n    # Laplace-smoothed mean reward, with global fallback.\n    if row and row[\'shown_count\']:\n        return (float(row[\'reward_sum\']) + 1.0) / (float(row[\'shown_count\']) + 2.0)\n    if global_row and global_row[\'shown_count\']:\n        return (float(global_row[\'reward_sum\']) + 1.0) / (float(global_row[\'shown_count\']) + 2.0)\n    return 0.5\n\n\ndef update_bandit(action_id: str, user: Dict[str, Any], reward: float) -> None:\n    segment = segment_for_user(user)\n    with get_conn() as conn:\n        existing = conn.execute(\'SELECT shown_count, reward_sum FROM action_rewards WHERE segment = ? AND action_id = ?\', (segment, action_id)).fetchone()\n        if existing:\n            conn.execute(\n                \'UPDATE action_rewards SET shown_count=?, reward_sum=?, last_updated=? WHERE segment=? AND action_id=?\',\n                (int(existing[\'shown_count\']) + 1, float(existing[\'reward_sum\']) + reward, utc_now(), segment, action_id)\n            )\n        else:\n            conn.execute(\n                \'INSERT INTO action_rewards VALUES (?, ?, ?, ?, ?)\',\n                (segment, action_id, 1, reward, utc_now())\n            )\n        conn.commit()\n\n\ndef get_recent_social_topics(limit: int = 100) -> Dict[str, int]:\n    posts = query_df(\'SELECT tags_json FROM posts ORDER BY created_at DESC LIMIT ?\', (limit,))\n    collabs = query_df(\'SELECT topics_json AS tags_json FROM collaborations ORDER BY created_at DESC LIMIT ?\', (limit,))\n    topic_counts: Dict[str, int] = {t: 0 for t in TOPICS}\n    for df in [posts, collabs]:\n        if df.empty:\n            continue\n        for raw in df[\'tags_json\'].tolist():\n            for topic in safe_json_loads(raw, []):\n                if topic in topic_counts:\n                    topic_counts[topic] += 1\n    return topic_counts\n\n\ndef action_probabilities(model: Optional[Pipeline], user_features: pd.DataFrame) -> Dict[str, float]:\n    if model is None:\n        return {}\n    try:\n        probs = model.predict_proba(user_features)[0]\n        labels = list(model.named_steps[\'model\'].classes_)\n        return dict(zip(labels, [float(p) for p in probs]))\n    except Exception:\n        return {}\n\n\ndef recommend_actions(user: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], str]:\n    consent_ai = bool(user.get(\'consent_ai\'))\n    rf, mlp, labels, status = train_models() if consent_ai else (None, None, [], \'AI personalization is off until the user opts in.\')\n    user_features = features_for_current_user(user)\n    rf_probs = action_probabilities(rf, user_features) if consent_ai else {}\n    mlp_probs = action_probabilities(mlp, user_features) if consent_ai else {}\n    social_counts = get_recent_social_topics()\n    max_social = max(social_counts.values()) if social_counts else 1\n    max_social = max(max_social, 1)\n\n    interests = set(user.get(\'interests\', []))\n    goals = set(user.get(\'goals\', []))\n    knowledge = user.get(\'knowledge_level\', \'Beginner\')\n    scored: List[Dict[str, Any]] = []\n\n    for action in ACTIONS:\n        tags = set(action[\'tags\'])\n        interest_overlap = len(tags.intersection(interests)) / max(len(tags), 1)\n        goal_bonus = 0.0\n        if action[\'type\'] == \'learn\' and \'Learn fundamentals\' in goals:\n            goal_bonus += 0.12\n        if action[\'type\'] == \'community\' and (\'Share outreach content\' in goals or \'Ask questions\' in goals):\n            goal_bonus += 0.14\n        if action[\'type\'] == \'collaboration\' and \'Find research collaborators\' in goals:\n            goal_bonus += 0.18\n        if action[\'action_id\'] == \'learn_ai_fusion\' and \'Build AI/ML skills\' in goals:\n            goal_bonus += 0.18\n        level_bonus = 0.0\n        if knowledge == \'Beginner\' and action[\'item_id\'] in {\'fusion_intro\', \'plasma_basics\', \'quiz\'}:\n            level_bonus += 0.12\n        if knowledge == \'Intermediate\' and action[\'item_id\'] in {\'tokamak_overview\', \'stellarator_overview\', \'inertial_fusion\', \'ai_fusion\'}:\n            level_bonus += 0.08\n        if knowledge == \'Advanced\' and action[\'item_id\'] in {\'lawson_q\', \'materials_tritium\', \'new_collab\'}:\n            level_bonus += 0.10\n        rule_score = min(1.0, 0.25 + 0.45 * interest_overlap + goal_bonus + level_bonus)\n        supervised_score = rf_probs.get(action[\'action_id\'], 0.0)\n        deep_score = mlp_probs.get(action[\'action_id\'], 0.0)\n        rl_score = bandit_score(action[\'action_id\'], user) if consent_ai else 0.5\n        social_score = sum(social_counts.get(t, 0) for t in tags) / (max_social * max(len(tags), 1))\n        social_score = min(1.0, social_score)\n\n        if consent_ai and rf_probs:\n            final_score = 0.38 * rule_score + 0.25 * supervised_score + 0.15 * deep_score + 0.15 * rl_score + 0.07 * social_score\n        else:\n            final_score = 0.70 * rule_score + 0.20 * rl_score + 0.10 * social_score\n\n        explanation_parts = []\n        if tags.intersection(interests):\n            explanation_parts.append(\'matches your selected interests\')\n        if goal_bonus:\n            explanation_parts.append(\'supports your stated goal\')\n        if supervised_score > 0.05:\n            explanation_parts.append(\'similar users gave positive feedback\')\n        if rl_score > 0.55:\n            explanation_parts.append(\'feedback rewards are trending positive\')\n        if social_score > 0.2:\n            explanation_parts.append(\'active community discussion exists\')\n        if not explanation_parts:\n            explanation_parts.append(action[\'why\'])\n\n        scored.append({\n            **action,\n            \'score\': final_score,\n            \'rule_score\': rule_score,\n            \'supervised_score\': supervised_score,\n            \'deep_score\': deep_score,\n            \'rl_score\': rl_score,\n            \'social_score\': social_score,\n            \'explanation\': \'; \'.join(explanation_parts)\n        })\n\n    # Simple exploration: sometimes surface a high-potential action outside the top purely exploitative list.\n    scored = sorted(scored, key=lambda x: x[\'score\'], reverse=True)\n    if consent_ai and len(scored) > 6 and random.random() < 0.15:\n        exploratory = random.choice(scored[6:])\n        exploratory[\'explanation\'] = \'exploration recommendation: testing a new path so the platform can learn from feedback\'\n        scored = scored[:4] + [exploratory]\n    else:\n        scored = scored[:5]\n\n    for action in scored:\n        log_event(user[\'user_id\'], \'recommendation_seen\', action[\'item_id\'], \',\'.join(action[\'tags\']), {\'action_id\': action[\'action_id\'], \'score\': action[\'score\']})\n    return scored, status\n\n\ndef get_module(module_id: str) -> Optional[Dict[str, Any]]:\n    for module in LEARNING_MODULES:\n        if module[\'id\'] == module_id:\n            return module\n    return None\n\n\ndef add_feedback(user: Dict[str, Any], action_id: str, suggested_item: str, rating: int, accepted: bool, reason: str) -> None:\n    run_sql(\n        \'INSERT INTO feedback VALUES (?, ?, ?, ?, ?, ?, ?, ?)\',\n        (str(uuid.uuid4()), user[\'user_id\'], action_id, suggested_item, int(rating), int(accepted), reason, utc_now())\n    )\n    reward = 1.0 if accepted else 0.0\n    reward = max(0.0, min(1.0, (float(rating) - 1.0) / 4.0 if rating else reward))\n    update_bandit(action_id, user, reward)\n    log_event(user[\'user_id\'], \'feedback_given\', suggested_item, \'\', {\'action_id\': action_id, \'rating\': rating, \'accepted\': accepted})\n\n\ndef render_header() -> None:\n    st.set_page_config(page_title=APP_NAME, page_icon=\'⚛️\', layout=\'wide\')\n    st.markdown(\n        """\n        <style>\n        .metric-card {border: 1px solid rgba(120,120,120,.25); border-radius: 16px; padding: 1rem; background: rgba(120,120,120,.05);}\n        .small-muted {font-size: 0.9rem; opacity: 0.75;}\n        .tag {display: inline-block; padding: .15rem .45rem; border-radius: 999px; border: 1px solid rgba(120,120,120,.35); margin: .1rem; font-size: .8rem;}\n        </style>\n        """,\n        unsafe_allow_html=True,\n    )\n    st.title(\'⚛️ FusionConnect AI\')\n    st.caption(\'An AI-assisted public learning, community, and collaboration platform for physics and nuclear fusion.\')\n    st.caption(f\'Founder & Author: {FOUNDER_NAME}\')\n    st.caption(f\'Advisor: {ADVISOR_NAME}\')\n\n\ndef sidebar_user(user_id: str) -> Dict[str, Any]:\n    user = get_user(user_id)\n    with st.sidebar:\n        st.subheader(\'Project Team\')\n        st.markdown(f\'**Founder & Author:** {FOUNDER_NAME}\')\n        st.markdown(f\'**Advisor:** {ADVISOR_NAME}\')\n        st.divider()\n        st.subheader(\'Anonymous session\')\n        st.caption(\'This prototype uses an anonymous ID. For a production launch, connect a real auth provider and external database.\')\n        st.code(user_id[:8] + \'...\' + user_id[-6:])\n        if st.button(\'Generate new anonymous session\'):\n            st.session_state.user_id = str(uuid.uuid4())\n            st.rerun()\n        st.divider()\n        st.markdown(f\'**Display name:** {user.get("display_name") or make_demo_name(user_id)}\')\n        st.markdown(f\'**Role:** {user.get("role") or "Not set"}\')\n        st.markdown(f\'**AI personalization:** {"On" if user.get("consent_ai") else "Off"}\')\n    return user\n\n\ndef page_start(user: Dict[str, Any]) -> None:\n    st.header(\'Start / Onboarding\')\n    st.write(\'Create a privacy-aware profile so the AI mentor can recommend learning modules, discussions, and collaboration paths.\')\n    st.info(\'Avoid collecting personally identifying information from students or minors unless you have the right consent process. Use broad, optional categories instead of exact birth date, address, or sensitive personal data.\')\n\n    with st.form(\'profile_form\'):\n        display_name = st.text_input(\'Public display name\', value=user.get(\'display_name\') or make_demo_name(user[\'user_id\']), max_chars=40)\n        col1, col2, col3 = st.columns(3)\n        with col1:\n            role = st.selectbox(\'Role\', ROLES, index=ROLES.index(user.get(\'role\')) if user.get(\'role\') in ROLES else 0)\n            age_band = st.selectbox(\'Age band (optional)\', AGE_BANDS, index=AGE_BANDS.index(user.get(\'age_band\')) if user.get(\'age_band\') in AGE_BANDS else 0)\n        with col2:\n            education_level = st.selectbox(\'Education level\', EDUCATION_LEVELS, index=EDUCATION_LEVELS.index(user.get(\'education_level\')) if user.get(\'education_level\') in EDUCATION_LEVELS else 0)\n            knowledge_level = st.selectbox(\'Physics/fusion knowledge\', KNOWLEDGE_LEVELS, index=KNOWLEDGE_LEVELS.index(user.get(\'knowledge_level\')) if user.get(\'knowledge_level\') in KNOWLEDGE_LEVELS else 0)\n        with col3:\n            region = st.text_input(\'Region or community (optional)\', value=user.get(\'region\') or \'Prefer not to say\', max_chars=80)\n        interests = st.multiselect(\'Physics and fusion interests\', TOPICS, default=[x for x in user.get(\'interests\', []) if x in TOPICS] or [\'fusion basics\'])\n        goals = st.multiselect(\'What do you want from this platform?\', GOALS, default=[x for x in user.get(\'goals\', []) if x in GOALS] or [\'Learn fundamentals\'])\n        consent_ai = st.checkbox(\'I agree to use my anonymous profile and app behavior for AI personalization inside this app.\', value=bool(user.get(\'consent_ai\')))\n        consent_public = st.checkbox(\'I understand posts/collaboration proposals may be visible to other users of this public app.\', value=bool(user.get(\'consent_public\')))\n        consent_research = st.checkbox(\'Optional: allow aggregated, anonymous usage statistics to support outreach/research reports.\', value=bool(user.get(\'consent_research\')))\n        submitted = st.form_submit_button(\'Save profile\')\n    if submitted:\n        if not interests:\n            st.warning(\'Please select at least one interest.\')\n            return\n        if not goals:\n            st.warning(\'Please select at least one goal.\')\n            return\n        profile = {\n            \'display_name\': display_name.strip() or make_demo_name(user[\'user_id\']),\n            \'role\': role,\n            \'age_band\': age_band,\n            \'education_level\': education_level,\n            \'region\': region.strip() or \'Prefer not to say\',\n            \'knowledge_level\': knowledge_level,\n            \'interests\': interests,\n            \'goals\': goals,\n            \'consent_ai\': consent_ai,\n            \'consent_public\': consent_public,\n            \'consent_research\': consent_research,\n        }\n        update_user_profile(user[\'user_id\'], profile)\n        st.success(\'Profile saved. Go to AI Mentor to see personalized suggestions.\')\n        st.rerun()\n\n    st.subheader(\'Prototype architecture\')\n    c1, c2, c3 = st.columns(3)\n    with c1:\n        st.markdown(\'**Supervised ML**\')\n        st.caption(\'Learns from anonymous profile + behavior + positive feedback to predict helpful next actions.\')\n    with c2:\n        st.markdown(\'**Deep neural network**\')\n        st.caption(\'Uses an MLP classifier as a lightweight neural model for nonlinear recommendation patterns.\')\n    with c3:\n        st.markdown(\'**Reinforcement learning**\')\n        st.caption(\'Uses reward feedback from users to improve future suggestions by segment.\')\n\n\ndef page_ai_mentor(user: Dict[str, Any]) -> None:\n    st.header(\'AI Mentor\')\n    if not user.get(\'consent_ai\'):\n        st.warning(\'AI personalization is currently off. Recommendations below use only basic rules and community trends. Enable personalization in Start / Onboarding for the full ML + neural + feedback pipeline.\')\n    recommendations, status = recommend_actions(user)\n    st.caption(status)\n\n    for i, rec in enumerate(recommendations, start=1):\n        with st.container(border=True):\n            cols = st.columns([0.72, 0.28])\n            with cols[0]:\n                st.subheader(f\'{i}. {rec["label"]}\')\n                st.write(rec[\'why\'])\n                st.markdown(\' \'.join([f\'<span class="tag">{t}</span>\' for t in rec[\'tags\']]), unsafe_allow_html=True)\n                st.caption(f\'Why suggested: {rec["explanation"]}\')\n            with cols[1]:\n                st.metric(\'AI score\', f\'{rec["score"]:.2f}\')\n                st.caption(f\'Rule {rec["rule_score"]:.2f} | ML {rec["supervised_score"]:.2f} | NN {rec["deep_score"]:.2f} | RL {rec["rl_score"]:.2f}\')\n                if st.button(\'Open / accept\', key=f\'accept_{rec["action_id"]}_{i}\'):\n                    add_feedback(user, rec[\'action_id\'], rec[\'item_id\'], rating=5, accepted=True, reason=\'accepted from recommendation card\')\n                    log_event(user[\'user_id\'], \'recommendation_accepted\', rec[\'item_id\'], \',\'.join(rec[\'tags\']), {\'action_id\': rec[\'action_id\']})\n                    if rec[\'type\'] == \'learn\' and rec[\'item_id\'] not in {\'quiz\'}:\n                        st.session_state.selected_module = rec[\'item_id\']\n                        st.session_state.nav_page = \'Learn Fusion\'\n                    elif rec[\'item_id\'] == \'new_post\':\n                        st.session_state.nav_page = \'Community Feed\'\n                    elif rec[\'item_id\'] in {\'new_collab\', \'collab_board\'}:\n                        st.session_state.nav_page = \'Collaboration Hub\'\n                    st.rerun()\n\n    st.subheader(\'Give feedback to train the AI\')\n    with st.form(\'feedback_form\'):\n        action_map = {f"{a[\'label\']} ({a[\'action_id\']})": a for a in ACTIONS}\n        chosen_label = st.selectbox(\'Suggestion you are rating\', list(action_map.keys()))\n        rating = st.slider(\'How useful was this suggestion?\', 1, 5, 4)\n        accepted = st.checkbox(\'I used or plan to use this suggestion\', value=rating >= 4)\n        reason = st.text_area(\'Optional feedback\', placeholder=\'Tell the platform what worked or did not work.\', max_chars=400)\n        submitted = st.form_submit_button(\'Submit feedback\')\n    if submitted:\n        action = action_map[chosen_label]\n        add_feedback(user, action[\'action_id\'], action[\'item_id\'], rating, accepted, reason)\n        st.success(\'Thanks. Your feedback updated the reinforcement-learning reward table and future recommendations.\')\n\n\ndef page_learn(user: Dict[str, Any]) -> None:\n    st.header(\'Learn Fusion\')\n    st.write(\'Use these modules as the first public-education content layer. Add more modules over time as the community grows.\')\n    col1, col2 = st.columns([0.35, 0.65])\n    with col1:\n        levels = [\'All\'] + KNOWLEDGE_LEVELS\n        selected_level = st.selectbox(\'Level filter\', levels)\n        selected_topic = st.selectbox(\'Topic filter\', [\'All\'] + TOPICS)\n        module_options = []\n        for module in LEARNING_MODULES:\n            if selected_level != \'All\' and module[\'level\'] != selected_level:\n                continue\n            if selected_topic != \'All\' and selected_topic not in module[\'tags\']:\n                continue\n            module_options.append(module)\n        default_module_id = st.session_state.get(\'selected_module\', module_options[0][\'id\'] if module_options else LEARNING_MODULES[0][\'id\'])\n        option_titles = {m[\'title\']: m for m in module_options or LEARNING_MODULES}\n        default_title = next((m[\'title\'] for m in option_titles.values() if m[\'id\'] == default_module_id), list(option_titles.keys())[0])\n        selected_title = st.radio(\'Choose a module\', list(option_titles.keys()), index=list(option_titles.keys()).index(default_title))\n        module = option_titles[selected_title]\n        if st.button(\'Mark module as viewed\'):\n            log_event(user[\'user_id\'], \'module_view\', module[\'id\'], \',\'.join(module[\'tags\']))\n            st.success(\'Progress saved.\')\n    with col2:\n        st.subheader(module[\'title\'])\n        st.caption(f"Level: {module[\'level\']} | Tags: {\', \'.join(module[\'tags\'])}")\n        st.write(module[\'summary\'])\n        st.markdown(module[\'content\'])\n        with st.expander(\'Active learning activity\'):\n            st.write(module[\'activity\'])\n        with st.expander(\'Quick quiz\'):\n            st.write(module[\'quiz_question\'])\n            answer = st.text_input(\'Your answer\', key=f\'quiz_{module["id"]}\')\n            if st.button(\'Check answer\', key=f\'check_{module["id"]}\'):\n                log_event(user[\'user_id\'], \'quiz_attempt\', module[\'id\'], \',\'.join(module[\'tags\']))\n                normalized = answer.lower().strip()\n                expected = module[\'quiz_answer\'].lower()\n                if any(word in normalized for word in expected.split()[:4]):\n                    st.success(\'Good direction. Compare with the model answer below.\')\n                else:\n                    st.info(\'Keep refining. Compare with the model answer below.\')\n                st.markdown(f\'**Model answer:** {module["quiz_answer"]}\')\n\n\ndef page_community(user: Dict[str, Any]) -> None:\n    st.header(\'Community Feed\')\n    st.write(\'A lightweight Reddit/X-style space for questions, ideas, outreach, and project notes.\')\n    if not user.get(\'consent_public\'):\n        st.info(\'You have not confirmed public posting consent. You can still read posts. To post, enable public posting consent in Start / Onboarding.\')\n\n    with st.expander(\'Create a post\', expanded=False):\n        with st.form(\'new_post_form\'):\n            category = st.selectbox(\'Category\', [\'Question\', \'Education\', \'AI / ML\', \'Research idea\', \'Outreach\', \'News discussion\'])\n            tags = st.multiselect(\'Tags\', TOPICS, default=[t for t in user.get(\'interests\', [])[:2] if t in TOPICS])\n            title = st.text_input(\'Title\', max_chars=140)\n            body = st.text_area(\'Post body\', max_chars=2500)\n            submitted = st.form_submit_button(\'Publish post\')\n        if submitted:\n            if not user.get(\'consent_public\'):\n                st.error(\'Enable public posting consent before publishing.\')\n            else:\n                ok, message = content_is_ok(title + \' \' + body)\n                if not ok:\n                    st.error(message)\n                elif not tags:\n                    st.error(\'Please choose at least one tag.\')\n                else:\n                    run_sql(\n                        \'INSERT INTO posts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)\',\n                        (str(uuid.uuid4()), user[\'user_id\'], user.get(\'display_name\') or make_demo_name(user[\'user_id\']), category, title.strip(), body.strip(), json_dumps(tags), 0, utc_now())\n                    )\n                    log_event(user[\'user_id\'], \'post_created\', topic=\',\'.join(tags), metadata={\'category\': category})\n                    st.success(\'Post published.\')\n                    st.rerun()\n\n    tag_filter = st.selectbox(\'Filter by tag\', [\'All\'] + TOPICS)\n    if tag_filter == \'All\':\n        posts = query_df(\'SELECT * FROM posts ORDER BY created_at DESC LIMIT 100\')\n    else:\n        posts = query_df(\'SELECT * FROM posts ORDER BY created_at DESC LIMIT 200\')\n        posts = posts[posts[\'tags_json\'].apply(lambda raw: tag_filter in safe_json_loads(raw, []))]\n\n    if posts.empty:\n        st.info(\'No posts yet. Be the first to start a discussion.\')\n    for _, post in posts.iterrows():\n        tags = safe_json_loads(post[\'tags_json\'], [])\n        with st.container(border=True):\n            st.subheader(post[\'title\'])\n            st.caption(f"{post[\'category\']} · by {post[\'display_name\']} · {post[\'created_at\']} · ▲ {int(post[\'upvotes\'])}")\n            st.markdown(\' \'.join([f\'<span class="tag">{t}</span>\' for t in tags]), unsafe_allow_html=True)\n            st.write(post[\'body\'])\n            c1, c2 = st.columns([0.18, 0.82])\n            with c1:\n                if st.button(\'▲ Upvote\', key=f\'up_{post["post_id"]}\'):\n                    run_sql(\'UPDATE posts SET upvotes = upvotes + 1 WHERE post_id = ?\', (post[\'post_id\'],))\n                    log_event(user[\'user_id\'], \'post_upvote\', post[\'post_id\'], \',\'.join(tags))\n                    st.rerun()\n            with c2:\n                with st.form(f\'comment_{post["post_id"]}\'):\n                    comment = st.text_input(\'Add a comment\', key=f\'comment_text_{post["post_id"]}\', max_chars=800)\n                    sent = st.form_submit_button(\'Comment\')\n                if sent:\n                    if not user.get(\'consent_public\'):\n                        st.error(\'Enable public posting consent before commenting.\')\n                    else:\n                        ok, message = content_is_ok(comment)\n                        if ok:\n                            run_sql(\'INSERT INTO comments VALUES (?, ?, ?, ?, ?, ?)\', (str(uuid.uuid4()), post[\'post_id\'], user[\'user_id\'], user.get(\'display_name\') or make_demo_name(user[\'user_id\']), comment.strip(), utc_now()))\n                            log_event(user[\'user_id\'], \'comment_created\', post[\'post_id\'], \',\'.join(tags))\n                            st.rerun()\n                        else:\n                            st.error(message)\n            comments = query_df(\'SELECT display_name, body, created_at FROM comments WHERE post_id = ? ORDER BY created_at ASC LIMIT 20\', (post[\'post_id\'],))\n            if not comments.empty:\n                with st.expander(f\'{len(comments)} comment(s)\'):\n                    for _, row in comments.iterrows():\n                        st.markdown(f\'**{row["display_name"]}** · {row["created_at"]}\')\n                        st.write(row[\'body\'])\n\n\ndef page_collaboration(user: Dict[str, Any]) -> None:\n    st.header(\'Collaboration Hub\')\n    st.write(\'Post mini-project ideas, class activities, reading groups, or research collaboration prompts.\')\n    with st.expander(\'Create collaboration proposal\', expanded=False):\n        with st.form(\'collab_form\'):\n            title = st.text_input(\'Proposal title\', max_chars=160)\n            topics = st.multiselect(\'Topics\', TOPICS, default=[t for t in user.get(\'interests\', [])[:3] if t in TOPICS])\n            summary = st.text_area(\'Summary\', max_chars=2500, placeholder=\'What is the project? What would collaborators do? What is the first milestone?\')\n            skills_needed = st.text_input(\'Skills needed\', max_chars=250, placeholder=\'Example: Python, physics teaching, plasma diagnostics, literature review\')\n            contact_hint = st.text_input(\'Contact hint (optional)\', max_chars=200, placeholder=\'Example: reply in comments, school club, public email, Discord handle\')\n            status = st.selectbox(\'Status\', [\'Open\', \'Planning\', \'Needs mentor\', \'Classroom activity\', \'Research idea\'])\n            submitted = st.form_submit_button(\'Publish collaboration proposal\')\n        if submitted:\n            if not user.get(\'consent_public\'):\n                st.error(\'Enable public posting consent before publishing.\')\n            else:\n                ok, message = content_is_ok(title + \' \' + summary)\n                if not ok:\n                    st.error(message)\n                elif not topics:\n                    st.error(\'Please choose at least one topic.\')\n                else:\n                    run_sql(\n                        \'INSERT INTO collaborations VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\',\n                        (str(uuid.uuid4()), user[\'user_id\'], user.get(\'display_name\') or make_demo_name(user[\'user_id\']), title.strip(), summary.strip(), json_dumps(topics), skills_needed.strip(), contact_hint.strip(), status, utc_now())\n                    )\n                    log_event(user[\'user_id\'], \'collab_created\', topic=\',\'.join(topics), metadata={\'status\': status})\n                    st.success(\'Collaboration proposal published.\')\n                    st.rerun()\n\n    proposals = query_df(\'SELECT * FROM collaborations ORDER BY created_at DESC LIMIT 100\')\n    if proposals.empty:\n        st.info(\'No collaboration proposals yet.\')\n    for _, row in proposals.iterrows():\n        topics = safe_json_loads(row[\'topics_json\'], [])\n        with st.container(border=True):\n            st.subheader(row[\'title\'])\n            st.caption(f"{row[\'status\']} · by {row[\'display_name\']} · {row[\'created_at\']}")\n            st.markdown(\' \'.join([f\'<span class="tag">{t}</span>\' for t in topics]), unsafe_allow_html=True)\n            st.write(row[\'summary\'])\n            if row[\'skills_needed\']:\n                st.markdown(f\'**Skills needed:** {row["skills_needed"]}\')\n            if row[\'contact_hint\']:\n                st.markdown(f\'**Contact hint:** {row["contact_hint"]}\')\n            if st.button(\'I am interested\', key=f\'interested_{row["collab_id"]}\'):\n                log_event(user[\'user_id\'], \'collab_interest\', row[\'collab_id\'], \',\'.join(topics))\n                st.success(\'Interest recorded. In a production version, this would notify the proposal owner.\')\n\n\ndef qr_png_bytes(url: str) -> bytes:\n    qr = qrcode.QRCode(version=1, box_size=8, border=3)\n    qr.add_data(url)\n    qr.make(fit=True)\n    img = qr.make_image(fill_color=\'black\', back_color=\'white\')\n    buffer = io.BytesIO()\n    img.save(buffer, format=\'PNG\')\n    return buffer.getvalue()\n\n\ndef page_dashboard(user: Dict[str, Any]) -> None:\n    st.header(\'Founder Dashboard\')\n    st.write(\'This page helps Ethan Meline track anonymized launch, outreach, and community-growth metrics.\')\n    default_pass = \'demo\'\n    try:\n        admin_pass = st.secrets.get(\'ADMIN_PASSCODE\', default_pass)\n    except Exception:\n        admin_pass = default_pass\n    password = st.text_input(\'Admin passcode\', type=\'password\', help=\'Default for this prototype is “demo”. Change ADMIN_PASSCODE in Streamlit secrets for deployment.\')\n    if password != admin_pass:\n        st.info(\'Enter the admin passcode to view dashboard metrics.\')\n        return\n\n    users = query_df(\'SELECT * FROM users\')\n    events = query_df(\'SELECT * FROM events\')\n    posts = query_df(\'SELECT * FROM posts\')\n    feedback = query_df(\'SELECT * FROM feedback\')\n    collabs = query_df(\'SELECT * FROM collaborations\')\n\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric(\'Anonymous signups\', len(users[~users[\'user_id\'].astype(str).str.startswith(\'seed-user\')]) if not users.empty else 0)\n    c2.metric(\'Community posts\', len(posts))\n    c3.metric(\'Collaboration proposals\', len(collabs))\n    c4.metric(\'AI feedback records\', len(feedback))\n\n    if not events.empty:\n        events[\'date\'] = pd.to_datetime(events[\'created_at\'], errors=\'coerce\').dt.date\n        growth = events[events[\'event_type\'].eq(\'signup\')].groupby(\'date\').size().reset_index(name=\'signups\')\n        if not growth.empty:\n            st.subheader(\'Signup trend\')\n            st.line_chart(growth.set_index(\'date\'))\n\n    col1, col2 = st.columns(2)\n    with col1:\n        st.subheader(\'User role distribution\')\n        if not users.empty:\n            role_counts = users[~users[\'user_id\'].astype(str).str.startswith(\'seed-user\')][\'role\'].fillna(\'Unknown\').value_counts().reset_index()\n            role_counts.columns = [\'role\', \'count\']\n            if not role_counts.empty:\n                st.plotly_chart(px.bar(role_counts, x=\'role\', y=\'count\'), use_container_width=True)\n            else:\n                st.info(\'No real user roles yet.\')\n    with col2:\n        st.subheader(\'Interest distribution\')\n        topic_counts = {topic: 0 for topic in TOPICS}\n        if not users.empty:\n            for raw in users[~users[\'user_id\'].astype(str).str.startswith(\'seed-user\')][\'interests_json\'].fillna(\'[]\'):\n                for topic in safe_json_loads(raw, []):\n                    if topic in topic_counts:\n                        topic_counts[topic] += 1\n        topics_df = pd.DataFrame({\'topic\': list(topic_counts.keys()), \'count\': list(topic_counts.values())}).sort_values(\'count\', ascending=False)\n        st.plotly_chart(px.bar(topics_df.head(10), x=\'count\', y=\'topic\', orientation=\'h\'), use_container_width=True)\n\n    st.subheader(\'Referral / QR acquisition\')\n    if not users.empty and \'referral_source\' in users.columns:\n        referrals = users[~users[\'user_id\'].astype(str).str.startswith(\'seed-user\')][\'referral_source\'].fillna(\'organic\').replace(\'\', \'organic\').value_counts().reset_index()\n        referrals.columns = [\'source\', \'signups\']\n        if not referrals.empty:\n            st.plotly_chart(px.bar(referrals, x=\'source\', y=\'signups\'), use_container_width=True)\n        else:\n            st.info(\'No referral data yet.\')\n\n    st.subheader(\'QR code generator\')\n    app_url = st.text_input(\'Public app URL\', placeholder=\'https://your-app-name.streamlit.app\')\n    ref_code = st.text_input(\'Referral code for QR analytics\', value=\'ethan_fusion_outreach\')\n    if app_url:\n        final_url = app_url.strip()\n        separator = \'&\' if \'?\' in final_url else \'?\'\n        final_url = f\'{final_url}{separator}ref={ref_code.strip() or "ethan"}\'\n        png = qr_png_bytes(final_url)\n        st.image(png, caption=final_url, width=260)\n        st.download_button(\'Download QR code PNG\', data=png, file_name=\'fusionconnect_ai_qr.png\', mime=\'image/png\')\n\n    st.subheader(\'Export anonymized analytics\')\n    export = {\n        \'generated_at\': utc_now(),\n        \'anonymous_user_count\': int(len(users[~users[\'user_id\'].astype(str).str.startswith(\'seed-user\')])) if not users.empty else 0,\n        \'post_count\': int(len(posts)),\n        \'collaboration_count\': int(len(collabs)),\n        \'feedback_count\': int(len(feedback)),\n    }\n    st.download_button(\'Download metrics JSON\', json_dumps(export), file_name=\'fusionconnect_metrics.json\', mime=\'application/json\')\n\n\ndef page_privacy(user: Dict[str, Any]) -> None:\n    st.header(\'Privacy / Data\')\n    st.write(\'This prototype is designed around anonymous IDs, optional broad profile fields, and transparent personalization.\')\n    st.markdown(\n        """\n**Recommended production upgrades before public launch:**\n\n- Use real authentication if users need persistent identity.\n- Use a managed database such as PostgreSQL, Supabase, or Firebase instead of local SQLite.\n- Add a real privacy policy, terms of use, consent flow for minors, and content-moderation process.\n- Add role-based admin access, audit logs, and data-retention controls.\n- Do not collect exact birth dates, home addresses, or sensitive demographics unless legally necessary and properly protected.\n"""\n    )\n    user_rows = query_df(\'SELECT * FROM users WHERE user_id = ?\', (user[\'user_id\'],))\n    event_rows = query_df(\'SELECT * FROM events WHERE user_id = ?\', (user[\'user_id\'],))\n    feedback_rows = query_df(\'SELECT * FROM feedback WHERE user_id = ?\', (user[\'user_id\'],))\n    posts_rows = query_df(\'SELECT * FROM posts WHERE user_id = ?\', (user[\'user_id\'],))\n    collab_rows = query_df(\'SELECT * FROM collaborations WHERE user_id = ?\', (user[\'user_id\'],))\n    export = {\n        \'user\': user_rows.to_dict(orient=\'records\'),\n        \'events\': event_rows.to_dict(orient=\'records\'),\n        \'feedback\': feedback_rows.to_dict(orient=\'records\'),\n        \'posts\': posts_rows.to_dict(orient=\'records\'),\n        \'collaborations\': collab_rows.to_dict(orient=\'records\'),\n    }\n    st.download_button(\'Download my data JSON\', data=json_dumps(export), file_name=\'my_fusionconnect_data.json\', mime=\'application/json\')\n\n    st.subheader(\'Delete anonymous account data\')\n    st.warning(\'This deletes your current anonymous profile, events, feedback, posts, comments, and collaboration proposals from this local prototype database.\')\n    confirm = st.text_input(\'Type DELETE to confirm\')\n    if st.button(\'Delete my current anonymous data\'):\n        if confirm == \'DELETE\':\n            uid = user[\'user_id\']\n            with get_conn() as conn:\n                for table in [\'feedback\', \'events\', \'comments\', \'posts\', \'collaborations\', \'users\']:\n                    conn.execute(f\'DELETE FROM {table} WHERE user_id = ?\', (uid,))\n                conn.commit()\n            st.session_state.user_id = str(uuid.uuid4())\n            st.success(\'Deleted current anonymous data and started a new anonymous session.\')\n            st.rerun()\n        else:\n            st.error(\'Confirmation text did not match DELETE.\')\n\n\ndef main() -> None:\n    init_db()\n    ensure_seed_data()\n    render_header()\n    user_id = create_user_if_needed()\n    user = sidebar_user(user_id)\n\n    pages = [\'Start / Onboarding\', \'AI Mentor\', \'Learn Fusion\', \'Community Feed\', \'Collaboration Hub\', \'Founder Dashboard\', \'Privacy / Data\']\n    if \'nav_page\' in st.session_state and st.session_state.nav_page in pages:\n        default_index = pages.index(st.session_state.nav_page)\n        del st.session_state.nav_page\n    else:\n        default_index = 0\n    page = st.sidebar.radio(\'Navigate\', pages, index=default_index)\n\n    if page == \'Start / Onboarding\':\n        page_start(user)\n    elif page == \'AI Mentor\':\n        page_ai_mentor(user)\n    elif page == \'Learn Fusion\':\n        page_learn(user)\n    elif page == \'Community Feed\':\n        page_community(user)\n    elif page == \'Collaboration Hub\':\n        page_collaboration(user)\n    elif page == \'Founder Dashboard\':\n        page_dashboard(user)\n    elif page == \'Privacy / Data\':\n        page_privacy(user)\n\n    st.divider()\n    st.caption(f\'Founder & Author: {FOUNDER_NAME}  |  Advisor: {ADVISOR_NAME}\')\n    st.caption(\'Prototype only: educational content is simplified; use expert review before presenting it as authoritative curriculum. AI suggestions are recommendations, not scientific or academic advice.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'

Path('fusionconnect_ai_streamlit_app.py').write_text(app_code, encoding='utf-8')
print('Created fusionconnect_ai_streamlit_app.py')
print('Characters:', len(app_code))


In [ ]:

# 4) Validate that the generated app file compiles.
import py_compile
from pathlib import Path

app_file = Path('fusionconnect_ai_streamlit_app.py')
py_compile.compile(str(app_file), doraise=True)
print('Compile check passed:', app_file.resolve())
print('Project files:')
for path in ['fusionconnect_ai_streamlit_app.py', 'requirements.txt', '.streamlit/config.toml']:
    print(' -', path)


In [ ]:

# 5) Optional: run the Streamlit app inside Google Colab.
# After running this cell, copy the public URL printed by localtunnel.
# If localtunnel asks for a password, it is usually the Colab machine public IP printed below.

import subprocess, time, os, sys

print('Starting Streamlit on port 8501...')
process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', 'fusionconnect_ai_streamlit_app.py',
    '--server.port', '8501', '--server.address', '0.0.0.0'
])
time.sleep(5)
print('Streamlit process started. If it stops, check the cell output above.')

# Public tunnel through localtunnel. This requires Node/npm, which Colab normally provides.
print('
Launching localtunnel...')
os.system('npx --yes localtunnel --port 8501')


In [ ]:

# 6) Optional: create a zip package for GitHub / Streamlit Community Cloud deployment.
# Download this zip, upload the files to a GitHub repository, and set the entrypoint to fusionconnect_ai_streamlit_app.py.

from pathlib import Path
import zipfile

zip_name = 'fusionconnect_ai_streamlit_project.zip'
files_to_zip = ['fusionconnect_ai_streamlit_app.py', 'requirements.txt', '.streamlit/config.toml']
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        z.write(file)
print('Created', zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print('Not running in Colab, or download helper unavailable. File is available at:', Path(zip_name).resolve())



## Streamlit Community Cloud deployment checklist

1. Create a GitHub repository.
2. Upload these files from the notebook output:
   - `fusionconnect_ai_streamlit_app.py`
   - `requirements.txt`
   - `.streamlit/config.toml`
3. In Streamlit Community Cloud, create a new app from the GitHub repository.
4. Set the app entrypoint to `fusionconnect_ai_streamlit_app.py`.
5. In Advanced settings, choose a Python 3.12 runtime if available.
6. Add a secret named `ADMIN_PASSCODE` so the Founder Dashboard is not left with the demo passcode.
7. After deployment, copy the public app URL into the dashboard QR-code generator.

For a larger public launch, replace the local SQLite database with a persistent database service.


Project credits: **Founder & Author: Ethan Meline**; **Advisor: Dr. Qingyang Xiao**.
